# 🏥 Healthcare Intelligence System - Data Pipeline

## Overview
This notebook processes 10,000+ healthcare facility records from India, extracting structured and unstructured data for an agentic AI system.

## Pipeline Steps
1. **Load Data**: Excel → Spark DataFrame
2. **Clean & Normalize**: Parse JSON fields, handle missing values
3. **Create Delta Tables**: Store in Unity Catalog
4. **Build Capabilities Index**: Extract searchable capabilities
5. **Quality Validation**: Verify data integrity

## Output Tables
- `healthcare_intel.facilities.raw` - Original data
- `healthcare_intel.facilities.cleaned` - Processed data
- `healthcare_intel.facilities.capabilities` - Exploded capabilities for semantic search

In [0]:
print("Hello World!")

Hello World!


In [0]:
# Install required packages for Excel processing
%pip install openpyxl pandas==2.0.3 -q

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json

print("✅ Dependencies installed")

# ============================================================================
# LOAD EXCEL DATA
# ============================================================================
file_path = "/Workspace/Users/umarofficial56@gmail.com/VF_Hackathon_Dataset_India_Large.xlsx"

print(f"📂 Loading dataset from: {file_path}")

# Read Excel using pandas first (better Excel support)
pandas_df = pd.read_excel(file_path)

print(f"✅ Loaded {len(pandas_df)} records with {len(pandas_df.columns)} columns")


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
✅ Dependencies installed
📂 Loading dataset from: /Workspace/Users/umarofficial56@gmail.com/VF_Hackathon_Dataset_India_Large.xlsx
✅ Loaded 10000 records with 41 columns


In [0]:

# Fix mixed types - convert all object columns to strings to avoid Arrow conversion errors
for col in pandas_df.columns:
    if pandas_df[col].dtype == 'object':
        pandas_df[col] = pandas_df[col].astype(str).replace('nan', None)

print("✅ Type conversion completed")

# Convert to Spark DataFrame with explicit schema handling
df_raw = spark.createDataFrame(pandas_df)

# Show schema
print("\n📋 Schema Overview:")
df_raw.printSchema()

# Display sample
print("\n📊 Sample Records:")
display(df_raw.limit(3))

✅ Type conversion completed

📋 Schema Overview:
root
 |-- name: string (nullable = true)
 |-- phone_numbers: string (nullable = true)
 |-- officialPhone: double (nullable = true)
 |-- email: string (nullable = true)
 |-- websites: string (nullable = true)
 |-- officialWebsite: string (nullable = true)
 |-- yearEstablished: double (nullable = true)
 |-- facebookLink: string (nullable = true)
 |-- twitterLink: string (nullable = true)
 |-- linkedinLink: string (nullable = true)
 |-- instagramLink: string (nullable = true)
 |-- address_line1: string (nullable = true)
 |-- address_line2: string (nullable = true)
 |-- address_line3: string (nullable = true)
 |-- address_city: string (nullable = true)
 |-- address_stateOrRegion: string (nullable = true)
 |-- address_zipOrPostcode: string (nullable = true)
 |-- address_country: string (nullable = true)
 |-- address_countryCode: string (nullable = true)
 |-- facilityTypeId: string (nullable = true)
 |-- operatorTypeId: string (nullable = true)

name,phone_numbers,officialPhone,email,websites,officialWebsite,yearEstablished,facebookLink,twitterLink,linkedinLink,instagramLink,address_line1,address_line2,address_line3,address_city,address_stateOrRegion,address_zipOrPostcode,address_country,address_countryCode,facilityTypeId,operatorTypeId,affiliationTypeIds,description,numberDoctors,capacity,specialties,procedure,equipment,capability,recency_of_page_update,distinct_social_media_presence_count,affiliated_staff_presence,custom_logo_presence,number_of_facts_about_the_organization,post_metrics_most_recent_social_media_post_date,post_metrics_post_count,engagement_metrics_n_followers,engagement_metrics_n_likes,engagement_metrics_n_engagements,latitude,longitude
1000 Smiles Dental Clinic,"[""+919392803399""]",9.19392803399E11,1000smilesdentalclinic@gmail.com,"[""https://www.facebook.com/249768584883702"",""https://www.justdial.com/Hyderabad/1000-Smiles-Dental-Clinic-Opposie-Hdfc-Bank-Amberpet/040PXX40-XX40-160227012741-N9P1_BZDET""]",null,null,https://www.facebook.com/249768584883702,null,null,null,"Plot No 2-2-1075/15/1/5, Shivam Road","Opposite Hdfc Bank, Tilak Nagar","Near Old Mla Quarters, Hyderguda",Hyderabad,Telangana,500013,India,IN,clinic,null,null,"Dental clinic offering RCT (Root Canal) and Laser Dentistry in Amberpet, Hyderabad.",null,null,"[""familyMedicine"",""periodontics"",""endodontics"",""dentistry"",""aestheticDentistry""]","[""Performs root canal therapy (RCT)"",""Provides laser dentistry services""]",[],"[""Has been in operation for 10 years"",""Dental clinic""]",null,1.0,0.0,0.0,null,null,null,23.0,10.0,null,17.3977394104003,78.482681274414
108 Eye And Heath Centre,"[""+919015821616""]",9.19015821616E11,108healthcentre@gmail.com,"[""https://www.indiamart.com/108-eye-and-health-centre/"",""https://108-eye-health.localo.site/"",""108eye.com"",""https://www.justdial.com/Noida/108-Eye-And-Heath-Centre-Near-State-Bank-Of-India-D-Block-Sector-50-Noida-Sector-50/011PXX11-XX11-180216202406-Z2S8_BZDET"",""https://www.facebook.com/108eyeandhealthcentre/"",""https://108eye.com/"",""https://www.practo.com/noida/clinics/eye-clinics"",""https://www.facebook.com/252399348484757""]",108eye.com,null,https://www.facebook.com/252399348484757,null,null,null,House Number 108,"Near State Bank Of India D Block Sector 50, E Block",null,Noida,Uttar Pradesh,201307,India,IN,clinic,private,null,"108 Eye And Health Centre provides you the best range of treatments, treatment services, pharmaceutical injection, veterinary drugs & our services with effective & timely delivery.",1.0,null,"[""pediatricsAndStrabismusOphthalmology"",""ophthalmology"",""refractiveSurgeryOphthalmology"",""familyMedicine"",""eyeTraumaAndEmergencyEyeCare"",""cataractAndAnteriorSegmentSurgery"",""uveitisOphthalmology"",""retinaAndVitreoretinalOphthalmology"",""glaucomaOphthalmology"",""neuroOphthalmology""]","[""Treats Squint"",""Glaucoma management"",""Offers Paediatric Ophthalmology"",""Performs cataract surgery"",""General eye check-up"",""Manages Uveitis"",""Cataract surgery"",""LASIK"",""IPCL"",""Pre-LASIK tests"",""Provides Basic Eye Check Up"",""Offers Retina Treatment Services"",""Offers Glaucoma Treatment"",""Offers Neuro Ophthalmology"",""ICL"",""Post-LASIK care"",""Squint correction"",""Retina treatment"",""Pediatric ophthalmology services""]",[],"[""Neuro Ophthalmology services offered"",""Outpatient ophthalmology clinic"",""Has 1 ophthalmologist/eye surgeon on staff"",""Dr. B K Varma is an ophthalmologist/eye surgeon with 17 years of experience"",""Retina Treatment Services offered"",""Paediatric Ophthalmology services offered"",""Employs experienced ophthalmologists"",""Uses latest equipment and world-standard technologies in eye care"",""Provides comprehensive eye care services under one roof""]",null,3.0,1.0,1.0,null,null,null,null,null,null,28.57222366333,77.3690338134765
14 Stars Dental,"[""+918791556586""]",9.18791556586E11,doctor@14starsdental.com,"[""14starsdental.com"",""https://www.facebook

In [0]:
from pyspark.sql.functions import (
    col, when, trim, lower, regexp_replace, coalesce, 
    from_json, lit, array, struct, explode, concat_ws,
    round as spark_round, split, size, filter as array_filter
)
from pyspark.sql.types import ArrayType, StringType, StructType, StructField

print("🧹 Starting data cleaning and normalization...\n")

# ============================================================================
# 1. PARSE JSON FIELDS
# ============================================================================
print("1️⃣ Parsing JSON fields (specialties, procedure, equipment, capability)...")

# Define UDF to safely parse JSON strings
def safe_parse_json(json_str):
    if json_str is None or json_str == '':
        return []
    try:
        parsed = json.loads(json_str)
        return parsed if isinstance(parsed, list) else []
    except:
        return []

safe_parse_json_udf = udf(safe_parse_json, ArrayType(StringType()))

df_cleaned = df_raw \
    .withColumn("specialties_parsed", safe_parse_json_udf(col("specialties"))) \
    .withColumn("procedure_parsed", safe_parse_json_udf(col("procedure"))) \
    .withColumn("equipment_parsed", safe_parse_json_udf(col("equipment"))) \
    .withColumn("capability_parsed", safe_parse_json_udf(col("capability")))

print("   ✅ JSON fields parsed")

# ============================================================================
# 2. CLEAN TEXT FIELDS
# ============================================================================
print("2️⃣ Cleaning text fields...")

df_cleaned = df_cleaned \
    .withColumn("name_cleaned", trim(col("name"))) \
    .withColumn("description_cleaned", trim(col("description"))) \
    .withColumn("email_cleaned", lower(trim(col("email")))) \
    .withColumn("facilityType", coalesce(col("facilityTypeId"), lit("unknown"))) \
    .withColumn("operatorType", coalesce(col("operatorTypeId"), lit("unknown")))

print("   ✅ Text fields cleaned")

# ============================================================================
# 3. NORMALIZE PHONE NUMBERS
# ============================================================================
print("3️⃣ Normalizing phone numbers...")

# Extract first valid phone from array
def extract_phone(phone_json):
    if phone_json is None or phone_json == '':
        return None
    try:
        phones = json.loads(phone_json)
        if isinstance(phones, list) and len(phones) > 0:
            # Remove special chars, keep digits
            phone = phones[0].replace('+', '').replace('-', '').replace(' ', '')
            return phone if len(phone) >= 10 else None
        return None
    except:
        return None

extract_phone_udf = udf(extract_phone, StringType())

df_cleaned = df_cleaned \
    .withColumn("phone_primary", extract_phone_udf(col("phone_numbers")))

print("   ✅ Phone numbers normalized")

# ============================================================================
# 4. BUILD FULL ADDRESS
# ============================================================================
print("4️⃣ Building complete address...")

df_cleaned = df_cleaned \
    .withColumn("address_full", 
        concat_ws(", ",
            col("address_line1"),
            col("address_line2"),
            col("address_line3"),
            col("address_city"),
            col("address_stateOrRegion"),
            col("address_zipOrPostcode")
        )
    ) \
    .withColumn("city", coalesce(trim(col("address_city")), lit("Unknown"))) \
    .withColumn("state", coalesce(trim(col("address_stateOrRegion")), lit("Unknown")))

print("   ✅ Address fields consolidated")

# ============================================================================
# 5. VALIDATE & CLEAN COORDINATES
# ============================================================================
print("5️⃣ Validating coordinates...")

df_cleaned = df_cleaned \
    .withColumn("latitude_valid", 
        when((col("latitude").isNotNull()) & 
             (col("latitude").between(6, 37)), 
             spark_round(col("latitude"), 6)
        ).otherwise(None)
    ) \
    .withColumn("longitude_valid",
        when((col("longitude").isNotNull()) & 
             (col("longitude").between(68, 98)),
             spark_round(col("longitude"), 6)
        ).otherwise(None)
    ) \
    .withColumn("has_valid_location", 
        col("latitude_valid").isNotNull() & col("longitude_valid").isNotNull()
    )

print("   ✅ Coordinates validated (India bounds: 6-37°N, 68-98°E)")

# ============================================================================
# 6. CREATE CAPABILITY SUMMARY
# ============================================================================
print("6️⃣ Creating capability summary...")

df_cleaned = df_cleaned \
    .withColumn("total_capabilities", 
        size(col("specialties_parsed")) + 
        size(col("procedure_parsed")) + 
        size(col("equipment_parsed")) + 
        size(col("capability_parsed"))
    ) \
    .withColumn("has_emergency", 
        array_contains(col("capability_parsed"), "emergency") |
        array_contains(col("specialties_parsed"), "emergencyMedicine")
    ) \
    .withColumn("has_surgery", 
        array_contains(col("procedure_parsed"), "surgery") |
        array_contains(col("specialties_parsed"), "surgery")
    )

print("   ✅ Capability summary created")

# ============================================================================
# 7. HANDLE MISSING VALUES
# ============================================================================
print("7️⃣ Handling missing values...")

df_cleaned = df_cleaned \
    .withColumn("numberDoctors", coalesce(col("numberDoctors"), lit(0)).cast("int")) \
    .withColumn("capacity", coalesce(col("capacity"), lit(0)).cast("int")) \
    .withColumn("yearEstablished", col("yearEstablished").cast("int"))

print("   ✅ Missing values handled\n")

print("✅ Data cleaning complete!")
print(f"📊 Processed {df_cleaned.count()} facilities")

# Show sample cleaned data
print("\n📋 Sample Cleaned Records:")
display(df_cleaned.select(
    "name_cleaned", "city", "state", 
    "specialties_parsed", "procedure_parsed",
    "latitude_valid", "longitude_valid",
    "total_capabilities", "has_emergency", "has_surgery"
).limit(5))

🧹 Starting data cleaning and normalization...

1️⃣ Parsing JSON fields (specialties, procedure, equipment, capability)...
   ✅ JSON fields parsed
2️⃣ Cleaning text fields...
   ✅ Text fields cleaned
3️⃣ Normalizing phone numbers...
   ✅ Phone numbers normalized
4️⃣ Building complete address...
   ✅ Address fields consolidated
5️⃣ Validating coordinates...
   ✅ Coordinates validated (India bounds: 6-37°N, 68-98°E)
6️⃣ Creating capability summary...
   ✅ Capability summary created
7️⃣ Handling missing values...
   ✅ Missing values handled

✅ Data cleaning complete!
📊 Processed 10000 facilities

📋 Sample Cleaned Records:


name_cleaned,city,state,specialties_parsed,procedure_parsed,latitude_valid,longitude_valid,total_capabilities,has_emergency,has_surgery
1000 Smiles Dental Clinic,Hyderabad,Telangana,"List(familyMedicine, periodontics, endodontics, dentistry, aestheticDentistry)","List(Performs root canal therapy (RCT), Provides laser dentistry services)",17.397739,78.482681,9,false,false
108 Eye And Heath Centre,Noida,Uttar Pradesh,"List(pediatricsAndStrabismusOphthalmology, ophthalmology, refractiveSurgeryOphthalmology, familyMedicine, eyeTraumaAndEmergencyEyeCare, cataractAndAnteriorSegmentSurgery, uveitisOphthalmology, retinaAndVitreoretinalOphthalmology, glaucomaOphthalmology, neuroOphthalmology)","List(Treats Squint, Glaucoma management, Offers Paediatric Ophthalmology, Performs cataract surgery, General eye check-up, Manages Uveitis, Cataract surgery, LASIK, IPCL, Pre-LASIK tests, Provides Basic Eye Check Up, Offers Retina Treatment Services, Offers Glaucoma Treatment, Offers Neuro Ophthalmology, ICL, Post-LASIK care, Squint correction, Retina treatment, Pediatric ophthalmology services)",28.572224,77.369034,38,false,false
14 Stars Dental,Aurangabad,Uttar Pradesh,"List(dentistry, generalDentistry)",List(),27.456034,77.704361,3,false,false
24x7 Family Clinix,Ghaziabad,Uttar Pradesh,List(familyMedicine),List(),28.701427,77.428604,1,false,false
3 E EYE CARE,Ahmedabad,Gujarat,List(familyMedicine),List(),23.002222,72.51783,1,false,false


## Improvements


In [0]:
# Combine all text fields into one searchable string
df = df_cleaned.withColumn("all_text_lower",
    lower(concat_ws(" ",
        col("description_cleaned"),
        concat_ws(" ", col("specialties_parsed")),
        concat_ws(" ", col("procedure_parsed")),
        concat_ws(" ", col("equipment_parsed")),
        concat_ws(" ", col("capability_parsed"))
    ))
)

# Substring matching instead of exact matching
df = df \
    .withColumn("has_emergency",
        col("all_text_lower").contains("emergency") |
        col("all_text_lower").contains("trauma") |
        col("all_text_lower").contains("casualty") |
        col("all_text_lower").contains("accident")) \
    .withColumn("has_surgery",
        col("all_text_lower").contains("surgery") |
        col("all_text_lower").contains("surgical") |
        col("all_text_lower").contains("operation theatre")) \
    .withColumn("has_icu",
        col("all_text_lower").contains("icu") |
        col("all_text_lower").contains("intensive care") |
        col("all_text_lower").contains("critical care")) \
    .withColumn("has_dialysis",
        col("all_text_lower").contains("dialysis") |
        col("all_text_lower").contains("renal") |
        col("all_text_lower").contains("nephrology")) \
    .withColumn("has_neonatal",
        col("all_text_lower").contains("neonatal") |
        col("all_text_lower").contains("nicu") |
        col("all_text_lower").contains("newborn"))

In [0]:
#  Build the combined_text field

df = df.withColumn("combined_text",
concat_ws(" | ",
concat(lit("Facility: "), col("name_cleaned")),
concat(lit("Type: "), col("facilityType")),
concat(lit("Location: "), col("city"), lit(", "), col("state"),
        lit(" - "), col("address_zipOrPostcode")),
concat(lit("Description: "), coalesce(col("description_cleaned"),
        lit("No description"))),
concat(lit("Specialties: "), concat_ws(", ", col("specialties_parsed"))),
concat(lit("Procedures: "), concat_ws(", ", col("procedure_parsed"))),
concat(lit("Equipment: "), concat_ws(", ", col("equipment_parsed"))),
concat(lit("Capabilities: "), concat_ws(", ", col("capability_parsed")))
)
)

In [0]:
#  Normalize specialties into standard categories Without this, medical desert analysis will count "periodontics" and "endodontics" as separate specialties instead of both being "Dentistry."

SPECIALTY_MAP = {
    "familyMedicine": "Family Medicine",
    "dentistry": "Dentistry",
     "generalDentistry": "Dentistry",
    "aestheticDentistry": "Dentistry",
     "periodontics": "Dentistry",
    "endodontics": "Dentistry",
    "ophthalmology": "Ophthalmology",
    "pediatricsAndStrabismusOphthalmology": "Ophthalmology",
    "refractiveSurgeryOphthalmology": "Ophthalmology",
    "cataractAndAnteriorSegmentSurgery": "Ophthalmology",
    "glaucomaOphthalmology": "Ophthalmology",
    "neuroOphthalmology": "Ophthalmology",
    "retinaAndVitreoretinalOphthalmology": "Ophthalmology",
    "uveitisOphthalmology": "Ophthalmology",
    "eyeTraumaAndEmergencyEyeCare": "Ophthalmology",
    "cardiology": "Cardiology",
    "oncology": "Oncology",
     "surgery": "Surgery",
    "nephrology": "Nephrology",
     "neurology": "Neurology",
    "orthopedics": "Orthopedics",
     "gynecology": "Gynecology",
    "emergencyMedicine": "Emergency Medicine",
    "dermatology": "Dermatology",
     "psychiatry": "Mental Health",
    "radiology": "Radiology",
     "anesthesiology": "Anesthesiology",
}

def normalize_specialties(specialties_list):
    if not specialties_list:
        return []
    normalized = set()
    for s in specialties_list:
        normalized.add(SPECIALTY_MAP.get(s, s))
    return list(normalized)

normalize_udf = udf(normalize_specialties, ArrayType(StringType()))

df = df.withColumn("specialties_normalized", 
    normalize_udf(col("specialties_parsed")))

In [0]:
#  Completeness score per facility:

df = df.withColumn("completeness_score",
    (when(col("description").isNotNull(), 1).otherwise(0) +
     when(size(col("specialties_parsed")) > 0, 1).otherwise(0) +
     when(size(col("procedure_parsed")) > 0, 1).otherwise(0) +
     when(size(col("equipment_parsed")) > 0, 1).otherwise(0) +
     when(col("numberDoctors") > 0, 1).otherwise(0) +
     when(col("capacity") > 0, 1).otherwise(0) +
     when(col("has_valid_location"), 1).otherwise(0) +
     when(col("phone_primary").isNotNull(), 1).otherwise(0)
    ) / lit(8) * 100
)

In [0]:
# Region-level aggregation for medical deserts:

# PIN code prefix (first 3 digits) = regional zone
df = df.withColumn("pin_region",
    substring(col("address_zipOrPostcode"), 1, 3))

# Aggregate by region for the heatmap
desert_analysis = df.groupBy("state", "pin_region").agg(
    count("*").alias("facility_count"),
    sum(when(col("has_emergency"), 1).otherwise(0)).alias("emergency_count"),
    sum(when(col("has_surgery"), 1).otherwise(0)).alias("surgery_count"),
    sum(when(col("has_icu"), 1).otherwise(0)).alias("icu_count"),
    sum(when(col("has_dialysis"), 1).otherwise(0)).alias("dialysis_count"),
    avg("completeness_score").alias("avg_completeness")
)

In [0]:
print(f"Total rows: {df.count()}")
# Should still be 10,000

Total rows: 10000


In [0]:
# This should NOT be zero anymore — Row 1 "108 Eye And Heath Centre" 
# has "cataractAndAnteriorSegmentSurgery" and "eyeTraumaAndEmergencyEyeCare"
print("Emergency facilities:", df.filter(col("has_emergency") == True).count())
print("Surgery facilities:", df.filter(col("has_surgery") == True).count())
print("ICU facilities:", df.filter(col("has_icu") == True).count())
print("Dialysis facilities:", df.filter(col("has_dialysis") == True).count())
print("Neonatal facilities:", df.filter(col("has_neonatal") == True).count())

# Spot check: find the eye centre and confirm it's now True
df.filter(col("name_cleaned").contains("108 Eye")).select(
    "name_cleaned", "has_emergency", "has_surgery"
).show(truncate=False)

Emergency facilities: 741
Surgery facilities: 2239
ICU facilities: 339
Dialysis facilities: 235
Neonatal facilities: 149
+------------------------+-------------+-----------+
|name_cleaned            |has_emergency|has_surgery|
+------------------------+-------------+-----------+
|108 Eye And Heath Centre|true         |true       |
+------------------------+-------------+-----------+



In [0]:
# No nulls or empty strings allowed
print("Null combined_text:", df.filter(col("combined_text").isNull()).count())
print("Empty combined_text:", df.filter(col("combined_text") == "").count())

# Check average length — should be substantial, not just "Facility: | Type: | ..."
from pyspark.sql.functions import length, avg
df.select(avg(length(col("combined_text"))).alias("avg_char_length")).show()

# Eyeball one record
df.select("combined_text").limit(1).show(truncate=False)

Null combined_text: 0
Empty combined_text: 0
+---------------+
|avg_char_length|
+---------------+
|        615.957|
+---------------+

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|combined_text                                                                                                                                                                                                                                                                                                                                                                                                                     

In [0]:
print("=== FINAL DATASET HEALTH CHECK ===")
print(f"Total facilities: {df.count()}")
print(f"Valid coordinates: {df.filter(col('has_valid_location')).count()}")
print(f"Has description: {df.filter(col('description_cleaned').isNotNull()).count()}")
print(f"Has specialties: {df.filter(size(col('specialties_parsed')) > 0).count()}")
print(f"Has procedures: {df.filter(size(col('procedure_parsed')) > 0).count()}")
print(f"Has equipment: {df.filter(size(col('equipment_parsed')) > 0).count()}")
print(f"Emergency capable: {df.filter(col('has_emergency')).count()}")
print(f"Surgery capable: {df.filter(col('has_surgery')).count()}")
print(f"ICU capable: {df.filter(col('has_icu')).count()}")
print(f"Unique states: {df.select('state').distinct().count()}")
print(f"Unique PIN regions: {df.select('pin_region').distinct().count()}")
print(f"Avg completeness: {df.select(avg('completeness_score')).first()[0]:.1f}%")

=== FINAL DATASET HEALTH CHECK ===
Total facilities: 10000
Valid coordinates: 10000
Has description: 9060
Has specialties: 10000
Has procedures: 3398
Has equipment: 1598
Emergency capable: 741
Surgery capable: 2239
ICU capable: 339
Unique states: 194
Unique PIN regions: 427
Avg completeness: 56.0%


In [0]:
# Use display() for Spark DataFrames in Databricks notebooks to show output.
from pyspark.sql.functions import length, avg, size, col

# Final dataset health check summary as a DataFrame
summary_df = df.selectExpr(
    "count(*) as total_facilities",
    "sum(cast(has_valid_location as int)) as valid_coordinates",
    "sum(cast(description_cleaned is not null as int)) as has_description",
    "sum(cast(size(specialties_parsed) > 0 as int)) as has_specialties",
    "sum(cast(size(procedure_parsed) > 0 as int)) as has_procedures",
    "sum(cast(size(equipment_parsed) > 0 as int)) as has_equipment",
    "sum(cast(has_emergency as int)) as emergency_capable",
    "sum(cast(has_surgery as int)) as surgery_capable",
    "sum(cast(has_icu as int)) as icu_capable",
    "sum(cast(has_dialysis as int)) as dialysis_capable",
    "sum(cast(has_neonatal as int)) as neonatal_capable",
    "count(distinct address_stateOrRegion) as unique_states",
    "count(distinct pin_region) as unique_pin_regions",
    "avg(completeness_score) as avg_completeness",
    "sum(cast(combined_text is not null as int)) as has_combined_text",
    "avg(length(combined_text)) as avg_combined_text_length"
)

display(summary_df)

total_facilities,valid_coordinates,has_description,has_specialties,has_procedures,has_equipment,emergency_capable,surgery_capable,icu_capable,dialysis_capable,neonatal_capable,unique_states,unique_pin_regions,avg_completeness,has_combined_text,avg_combined_text_length
10000,10000,9060,10000,3398,1598,741,2239,339,235,149,194,426,55.98,10000,615.957


In [0]:
df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("healthcare_facilities_cleaned")

print("✅ Saved to Delta table")

✅ Saved to Delta table


In [0]:
df.toPandas().to_csv("/Workspace/Users/umarofficial56@gmail.com/cleaned_dataset.csv", index=False)